# Credit Card Fraud: SMOTE Class-Imbalance Analysis

This notebook extends `02_Model_Selection.ipynb` while preserving its official training data, feature engineering, and chronological 70/30 validation design. It tests whether synthetic minority oversampling improves fraud detection for LightGBM and XGBoost.

SMOTE is applied **only to the earliest 70% development partition**. The latest 30% validation partition is never resampled, so all reported precision, recall, ROC-AUC, and PR-AUC values represent the naturally imbalanced fraud environment.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from imblearn.over_sampling import SMOTENC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, PrecisionRecallDisplay,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')
plt.style.use('seaborn-v0_8-whitegrid')

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = PROJECT_ROOT / 'Data' / 'raw' / 'fraud_train.csv'
RESULTS_PATH = PROJECT_ROOT / 'Data' / 'processed' / 'smote_model_comparison.csv'
BASELINE_PATH = PROJECT_ROOT / 'Data' / 'processed' / 'tuned_boosting_comparison.csv'

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f'Missing official training data: {TRAIN_PATH}')

**What this block does:** Imports the SMOTE, preprocessing, boosting, evaluation, and plotting tools. It fixes the random seed and defines paths for the official training data, the previous tuned baseline results, and this notebook's output.

## 1. Reuse the established features

The transformations match notebook 02: amount and population are log-transformed, transaction time is converted into linear and cyclic features, age is measured at the transaction date, and customer-to-merchant distance is calculated with the Haversine formula. Direct identifiers and target-derived features are excluded.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized great-circle distance between two coordinate pairs."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
    )
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))


def prepare_model_features(raw):
    data = raw.drop(
        columns=[column for column in raw.columns if column.startswith('Unnamed:')],
        errors='ignore',
    ).copy()
    transaction_time = pd.to_datetime(
        data['trans_date_trans_time'], format='mixed', dayfirst=True, errors='coerce'
    )
    birth_date = pd.to_datetime(
        data['dob'], format='mixed', dayfirst=True, errors='coerce'
    )

    features = pd.DataFrame(index=data.index)
    features['amt_log1p'] = np.log1p(data['amt'])
    features['city_pop_log1p'] = np.log1p(data['city_pop'])
    features['age_at_transaction'] = (transaction_time - birth_date).dt.days / 365.2425
    features['trans_hour'] = transaction_time.dt.hour
    features['trans_day_of_week'] = transaction_time.dt.dayofweek
    features['trans_month'] = transaction_time.dt.month
    features['is_weekend'] = transaction_time.dt.dayofweek.ge(5).astype('int8')
    features['is_night'] = transaction_time.dt.hour.between(0, 5).astype('int8')
    features['hour_sin'] = np.sin(2 * np.pi * transaction_time.dt.hour / 24)
    features['hour_cos'] = np.cos(2 * np.pi * transaction_time.dt.hour / 24)
    features['day_sin'] = np.sin(2 * np.pi * transaction_time.dt.dayofweek / 7)
    features['day_cos'] = np.cos(2 * np.pi * transaction_time.dt.dayofweek / 7)
    features['merchant_distance_km'] = haversine_km(
        data['lat'].to_numpy(), data['long'].to_numpy(),
        data['merch_lat'].to_numpy(), data['merch_long'].to_numpy(),
    )
    features['lat'] = data['lat']
    features['long'] = data['long']

    for column in ['category', 'gender', 'state']:
        features[column] = data[column].astype('string').str.strip().fillna('missing')

    return features, data['is_fraud'].astype('int8'), transaction_time


identifier_types = {'cc_num': 'string', 'zip': 'string', 'trans_num': 'string'}
official_train_raw = pd.read_csv(TRAIN_PATH, dtype=identifier_types, low_memory=False)

if len(official_train_raw) != 1_296_675:
    raise ValueError(
        f'Expected 1,296,675 official training rows, found {len(official_train_raw):,}.'
    )

X_all, y_all, transaction_time = prepare_model_features(official_train_raw)
print(f'Official training rows: {len(X_all):,}')
print(f'Fraud cases: {y_all.sum():,} ({y_all.mean():.3%})')

**What this block does:** Recreates the exact row-level predictors used in notebook 02 and loads the official `fraud_train.csv`. The row-count validation guards against accidentally running the experiment on the external test file or an incomplete dataset.

## 2. Keep the chronological 70/30 split

The earliest 70% of transactions form the development set and the latest 30% form the locked validation set. This simulates learning from the past and evaluating in a later period. The validation set is left at its original fraud prevalence.

In [ ]:
order = np.argsort(transaction_time.to_numpy(), kind='stable')
X_all = X_all.iloc[order].reset_index(drop=True)
y_all = y_all.iloc[order].reset_index(drop=True)
transaction_time = transaction_time.iloc[order].reset_index(drop=True)

split_index = int(len(X_all) * 0.70)
X_train, X_validation = X_all.iloc[:split_index], X_all.iloc[split_index:]
y_train, y_validation = y_all.iloc[:split_index], y_all.iloc[split_index:]
time_train = transaction_time.iloc[:split_index]
time_validation = transaction_time.iloc[split_index:]

split_summary = pd.DataFrame({
    'partition': ['Earliest 70%: development', 'Latest 30%: validation'],
    'rows': [len(X_train), len(X_validation)],
    'start': [time_train.min(), time_validation.min()],
    'end': [time_train.max(), time_validation.max()],
    'fraud_cases': [int(y_train.sum()), int(y_validation.sum())],
    'fraud_rate': [y_train.mean(), y_validation.mean()],
})
display(split_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
split_summary.set_index('partition')['fraud_rate'].mul(100).plot.bar(
    ax=axes[0], color=['#4C78A8', '#F58518']
)
axes[0].set_title('Natural Fraud Rate by Time Partition')
axes[0].set_xlabel('')
axes[0].set_ylabel('Fraud rate (%)')
axes[0].tick_params(axis='x', rotation=0)

split_summary.set_index('partition')[['rows', 'fraud_cases']].plot.bar(
    ax=axes[1], logy=True, color=['#72B7B2', '#E45756']
)
axes[1].set_title('Rows and Fraud Cases (Log Scale)')
axes[1].set_xlabel('')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
plt.show()

**What this block does:** Sorts every transaction by date and reproduces the 70/30 time split. The two plots expose both the natural fraud rate and the large difference between majority and minority counts before oversampling.

## 3. Training-only preprocessing and SMOTENC

Standard SMOTE interpolates every column numerically. That would create impossible fractional values across category, gender, and state. `SMOTENC` is the appropriate SMOTE variant for this mixed dataset: it interpolates continuous fields while treating the three encoded categorical fields as categories.

All imputers, scalers, and category encoders are fitted on the development set only. Three moderate ratios are tested:

- `0.05`: 1 synthetic/real fraud observation for every 20 non-fraud observations;
- `0.10`: 1 fraud observation for every 10 non-fraud observations;
- `0.25`: 1 fraud observation for every 4 non-fraud observations.

Forcing a 50/50 balance is intentionally avoided because it would create hundreds of thousands of synthetic fraud rows and can severely reduce precision.

In [ ]:
numeric_model_features = [
    'amt_log1p', 'city_pop_log1p', 'age_at_transaction', 'trans_hour',
    'trans_day_of_week', 'trans_month', 'is_weekend', 'is_night',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'merchant_distance_km', 'lat', 'long',
]
categorical_model_features = ['category', 'gender', 'state']

before_smote = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_model_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]), categorical_model_features),
], sparse_threshold=0)

X_train_intermediate = before_smote.fit_transform(X_train)
X_validation_intermediate = before_smote.transform(X_validation)

categorical_indices = list(range(
    len(numeric_model_features),
    len(numeric_model_features) + len(categorical_model_features),
))

after_smote = ColumnTransformer([
    ('numeric', 'passthrough', list(range(len(numeric_model_features)))),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_indices),
])

# Fit final one-hot categories on real training values. SMOTENC only reuses these
# categorical codes, so synthetic rows cannot introduce a new category.
after_smote.fit(X_train_intermediate)
X_validation_encoded = after_smote.transform(X_validation_intermediate)

print(f'Intermediate training matrix: {X_train_intermediate.shape}')
print(f'Encoded validation matrix: {X_validation_encoded.shape}')

**What this block does:** Fits numeric imputation/scaling and categorical ordinal encoding on the development period. It records which columns are categorical for `SMOTENC`, then prepares a one-hot encoder so the boosting models receive the same model-ready representation for every sampling ratio.

## 4. Compare SMOTE ratios with LightGBM and XGBoost

At this point, we will be using the two best model as defined by the 02 notebook, and by using different SMOTE ratio, XGBoost achieved: higher precision, higher F1, higher ROC-AUC, and higher PR-AUC
Notice that the lower SMOTE ratio performed the best, meaning just a small size of oversampling is enough to improve minority class learning.
- Increasing SMOTE percentage points
 would improve recall but hurt precision.
- Therefore, applying SMOTE might successfully improve class balance during training, but aggressively oversampling reduced overall ranking performance.

In [ ]:
def classification_metrics(model_name, sampling_ratio, model, X_fit, y_fit, X_eval, y_eval):
    train_prediction = model.predict(X_fit).astype('int8').ravel()
    prediction = model.predict(X_eval).astype('int8').ravel()
    probability = model.predict_proba(X_eval)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_eval, prediction).ravel()
    return {
        'Model Name': model_name,
        'SMOTE Ratio': sampling_ratio,
        'Resampled Rows': len(y_fit),
        'Resampled Fraud Rate': y_fit.mean(),
        'Train Accuracy': accuracy_score(y_fit, train_prediction),
        'Validation Accuracy': accuracy_score(y_eval, prediction),
        'Precision': precision_score(y_eval, prediction, zero_division=0),
        'Recall': recall_score(y_eval, prediction, zero_division=0),
        'F1 Score': f1_score(y_eval, prediction, zero_division=0),
        'ROC-AUC': roc_auc_score(y_eval, probability),
        'PR-AUC': average_precision_score(y_eval, probability),
        'False Positives': int(fp),
        'False Negatives': int(fn),
    }


sampling_ratios = [0.05, 0.10, 0.25]
result_rows = []
fitted_smote_models = {}
resampling_rows = []

for sampling_ratio in sampling_ratios:
    sampler = SMOTENC(
        categorical_features=categorical_indices,
        sampling_strategy=sampling_ratio,
        k_neighbors=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    X_resampled, y_resampled = sampler.fit_resample(X_train_intermediate, y_train)
    X_resampled_encoded = after_smote.transform(X_resampled)

    resampling_rows.append({
        'SMOTE Ratio': sampling_ratio,
        'Before Rows': len(y_train),
        'Before Fraud Cases': int(y_train.sum()),
        'Before Fraud Rate': y_train.mean(),
        'After Rows': len(y_resampled),
        'After Fraud Cases': int(y_resampled.sum()),
        'After Fraud Rate': y_resampled.mean(),
        'Synthetic Fraud Rows': int(y_resampled.sum() - y_train.sum()),
    })

    models = {
        'LightGBM + SMOTE': LGBMClassifier(
            n_estimators=124, learning_rate=0.05, num_leaves=31,
            max_depth=-1, min_child_samples=50, reg_alpha=0.0, reg_lambda=1.0,
            subsample=0.9, colsample_bytree=0.9, n_jobs=-1,
            random_state=RANDOM_STATE, verbosity=-1,
        ),
        'XGBoost + SMOTE': XGBClassifier(
            n_estimators=117, learning_rate=0.05, max_depth=8,
            min_child_weight=10, gamma=0.1, reg_lambda=5.0,
            subsample=0.9, colsample_bytree=0.9,
            objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
            n_jobs=-1, random_state=RANDOM_STATE,
        ),
    }

    for model_name, model in models.items():
        started = perf_counter()
        model.fit(X_resampled_encoded, y_resampled)
        model_key = f'{model_name} ({sampling_ratio:.0%})'
        fitted_smote_models[model_key] = model
        row = classification_metrics(
            model_name, sampling_ratio, model,
            X_resampled_encoded, y_resampled,
            X_validation_encoded, y_validation,
        )
        row['Fit Seconds'] = perf_counter() - started
        result_rows.append(row)
        print(
            f'Finished {model_key}: PR-AUC={row["PR-AUC"]:.4f}, '
            f'precision={row["Precision"]:.4f}, recall={row["Recall"]:.4f}'
        )

resampling_summary = pd.DataFrame(resampling_rows)
smote_results = (
    pd.DataFrame(result_rows)
      .sort_values(['PR-AUC', 'Recall'], ascending=False)
      .reset_index(drop=True)
)

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
smote_results.to_csv(RESULTS_PATH, index=False)
display(resampling_summary)
display(smote_results)

**What this block does:** Creates synthetic fraud observations at each candidate ratio, fits the tuned LightGBM and XGBoost structures without class weights, and evaluates every model on the untouched later-period validation set. It saves the complete comparison to `Data/processed/smote_model_comparison.csv`.

## 5. Visual comparison and baseline check

The first plot shows whether increasing SMOTE strength improves validation PR-AUC. The second shows the precision-recall trade-off at the default 0.50 threshold. A model is not better merely because recall increases: a large loss of precision means substantially more legitimate transactions would be flagged.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

for model_name, group in smote_results.sort_values('SMOTE Ratio').groupby('Model Name'):
    axes[0].plot(
        group['SMOTE Ratio'].mul(100), group['PR-AUC'],
        marker='o', linewidth=2, label=model_name,
    )
axes[0].axhline(
    y_validation.mean(), color='gray', linestyle='--',
    label='Random-ranking baseline',
)
axes[0].set_title('Validation PR-AUC by SMOTE Ratio')
axes[0].set_xlabel('Minority-to-majority ratio after SMOTE (%)')
axes[0].set_ylabel('PR-AUC')
axes[0].legend()

for model_name, group in smote_results.groupby('Model Name'):
    axes[1].plot(
        group['Recall'], group['Precision'], marker='o', linewidth=2,
        label=model_name,
    )
    for _, row in group.iterrows():
        axes[1].annotate(
            f'{row["SMOTE Ratio"]:.0%}',
            (row['Recall'], row['Precision']),
            textcoords='offset points', xytext=(5, 5), fontsize=9,
        )
axes[1].set_title('Default-Threshold Precision vs Recall')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for model_key, model in fitted_smote_models.items():
    probability = model.predict_proba(X_validation_encoded)[:, 1]
    PrecisionRecallDisplay.from_predictions(
        y_validation, probability, name=model_key, ax=ax,
    )
ax.set_title('Precision–Recall Curves: SMOTE Models')
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "figure-8.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

**What this block does:** Plots PR-AUC across SMOTE ratios, the default-threshold precision-recall trade-off, and the full precision-recall curves. These views reveal whether SMOTE improves ranking quality or simply shifts the number of predicted fraud cases.

In [ ]:
best_smote = smote_results.iloc[0]

comparison_rows = [{
    'Approach': f'{best_smote["Model Name"]} at {best_smote["SMOTE Ratio"]:.0%}',
    'PR-AUC': best_smote['PR-AUC'],
    'ROC-AUC': best_smote['ROC-AUC'],
    'Precision': best_smote['Precision'],
    'Recall': best_smote['Recall'],
    'F1 Score': best_smote['F1 Score'],
}]

if BASELINE_PATH.exists():
    baseline = pd.read_csv(BASELINE_PATH).sort_values('PR-AUC', ascending=False).iloc[0]
    comparison_rows.append({
        'Approach': f'Notebook 02 winner: {baseline["Model Name"]}',
        'PR-AUC': baseline['PR-AUC'],
        'ROC-AUC': baseline['ROC-AUC'],
        'Precision': baseline['Precision'],
        'Recall': baseline['Recall'],
        'F1 Score': baseline['F1 Score'],
    })

final_comparison = pd.DataFrame(comparison_rows).sort_values('PR-AUC', ascending=False)
display(final_comparison)

winner = final_comparison.iloc[0]
print(f'Recommended approach by locked-validation PR-AUC: {winner["Approach"]}')
print(f'PR-AUC: {winner["PR-AUC"]:.4f}')
print('Do not choose SMOTE solely because it creates a more balanced training table.')

**What this block does:** Selects the strongest SMOTE result and, when notebook 02's saved results are available, compares it directly with the previous class-weighted tuned winner. The final recommendation still follows locked-validation PR-AUC rather than the apparent balance of the training data.

## 6. Interpretation and next decision

- SMOTE changes only the model's learning data; it does **not** change how common fraud is in production.
- If the best SMOTE model exceeds notebook 02's tuned winner on validation PR-AUC, retain it as a challenger for final testing.
- If SMOTE raises recall but lowers PR-AUC or precision, the original class-weighted booster remains the better ranking model; threshold tuning can then pursue the required recall more efficiently.
- Keep the external `fraud_test.csv` untouched until the feature pipeline, sampling ratio, model, and operating-threshold method have been frozen.
- The next business step is threshold selection using fraud loss, false-alert review cost, customer friction, and investigation capacity—not selecting the default 0.50 threshold automatically.